# 05 · Modellvergleich & Analyse — Favorita

Vollständiger Vergleich aller Modelle:
- Metriken auf Val- und Test-Set
- Feature Importance (XGBoost & LightGBM)
- Tiefergehende Modellanalyse
- Fehleranalyse pro Store
- Residuenanalyse des besten Modells

## 0 · Imports & Setup

In [1]:
import os
import sys
import joblib
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error
import torch
torch.set_num_threads(1)
from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST, NHITS

sys.path.append(os.path.abspath('../03_src'))
from config import FINAL, TARGET_COL, FEATURE_COLS, EXOG_COLS, STAT_EXOG, HIST_EXOG, TRAIN_END, VAL_END, LOOKBACK, HORIZON, MODELL_ORDER, MODELL_COLORS, PRED_COLS, RESULTS, FILE_NAMES
from utilis import run_sarimax, run_prophet, run_xgb, run_lgbm, to_nf_format, compute_metrics, load_preds, nf_to_polars

sns.set_style('whitegrid')

print('Setup ✓')

Setup ✓


## 1 · Daten laden

In [2]:
df = pl.read_parquet(FINAL / 'final_dataset.parquet')

train     = df.filter(pl.col('date') <= TRAIN_END)
val       = df.filter((pl.col('date') > TRAIN_END) & (pl.col('date') <= VAL_END))
test      = df.filter(pl.col('date') > VAL_END)
train_val = df.filter(pl.col('date') <= VAL_END)

stores = sorted(df['store_nbr'].unique().to_list())

print(f'Train:     {train.shape}  {train["date"].min()} → {train["date"].max()}')
print(f'Val:       {val.shape}    {val["date"].min()} → {val["date"].max()}')
print(f'Test:      {test.shape}   {test["date"].min()} → {test["date"].max()}')
print(f'Stores:    {len(stores)}')

Train:     (70114, 37)  2013-01-29 → 2016-12-31
Val:       (7965, 37)    2017-01-01 → 2017-05-31
Test:      (4104, 37)   2017-06-01 → 2017-08-15
Stores:    54


In [3]:
# Val-Vorhersagen laden
val_preds = {m: load_preds(f'val_{m.lower()}.parquet') for m in MODELL_ORDER}
available = [m for m, df_m in val_preds.items() if df_m is not None]
print(f'Verfügbare Val-Modelle: {available}')

Verfügbare Val-Modelle: ['SARIMAX', 'Prophet', 'XGBoost', 'LightGBM', 'PatchTST', 'NHITS']


## 2 · Val-Set Metriken

In [4]:
metrics_val = []

for modell in available:
    preds_df = val_preds[modell]
    pred_col = PRED_COLS[modell]
    merged   = preds_df.drop_nulls(subset=[pred_col, 'y_true'])
    metrics_val.append(
        compute_metrics(merged['y_true'].to_numpy(), merged[pred_col].to_numpy(), modell, 'val')
    )

metrics_val_df = pd.DataFrame(metrics_val).set_index('modell').drop(columns='split')
print('=== Val-Set Metriken ===')
print(metrics_val_df.sort_values('MAE').round(2))

=== Val-Set Metriken ===
             MAE     RMSE   MAPE
modell                          
XGBoost    11.47    24.75   0.74
LightGBM   12.95    26.40   0.85
Prophet   204.26   304.09  12.68
NHITS     225.32   337.95  14.08
PatchTST  466.54   562.35  37.03
SARIMAX   817.62  1672.62  66.16


## 3 · Test-Set: Re-Run aller Modelle

# SARIMAX

In [ ]:
print('SARIMAX...')
test_sarimax = run_sarimax(train_val, test, stores=stores, exog_cols=EXOG_COLS)
test_sarimax.write_parquet(RESULTS / 'test_sarimax.parquet')
print('SARIMAX ✓')

SARIMAX...


SARIMAX:   0%|          | 0/54 [00:00<?, ?it/s]c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:836: ValueWarning: No sup

SARIMAX ✓


# Prophet

In [ ]:
print('Prophet...')
test_prophet = run_prophet(train_val, test, stores=stores, extra_regressors=['oil_price'])
test_prophet.write_parquet(RESULTS / 'test_prophet.parquet')
print('Prophet ✓')

Prophet...


Prophet Progress:   0%|          | 0/54 [00:00<?, ?it/s]17:18:48 - cmdstanpy - INFO - Chain [1] start processing
17:18:49 - cmdstanpy - INFO - Chain [1] done processing
Prophet Progress:   2%|▏         | 1/54 [00:01<01:24,  1.59s/it]17:18:50 - cmdstanpy - INFO - Chain [1] start processing
17:18:50 - cmdstanpy - INFO - Chain [1] done processing
Prophet Progress:   4%|▎         | 2/54 [00:02<00:55,  1.06s/it]17:18:50 - cmdstanpy - INFO - Chain [1] start processing
17:18:50 - cmdstanpy - INFO - Chain [1] done processing
Prophet Progress:   6%|▌         | 3/54 [00:02<00:42,  1.20it/s]17:18:51 - cmdstanpy - INFO - Chain [1] start processing
17:18:51 - cmdstanpy - INFO - Chain [1] done processing
Prophet Progress:   7%|▋         | 4/54 [00:03<00:36,  1.37it/s]17:18:51 - cmdstanpy - INFO - Chain [1] start processing
17:18:51 - cmdstanpy - INFO - Chain [1] done processing
Prophet Progress:   9%|▉         | 5/54 [00:03<00:32,  1.51it/s]17:18:52 - cmdstanpy - INFO - Chain [1] start processing
17

Prophet ✓


# XGBoost & LightGBM

In [ ]:
X_train_val = train_val.select(FEATURE_COLS).to_numpy()
y_train_val = train_val.select(TARGET_COL).to_numpy().ravel()
X_test      = test.select(FEATURE_COLS).to_numpy()
y_test      = test.select(TARGET_COL).to_numpy().ravel()

model_xgb,  _ = run_xgb(X_train_val, y_train_val, X_test, y_test)
model_lgbm, _ = run_lgbm(X_train_val, y_train_val, X_test, y_test)

test_xgb = (
    test.select(['store_nbr', 'date', TARGET_COL]).rename({TARGET_COL: 'y_true'})
    .with_columns(pl.Series('pred_xgb', model_xgb.predict(X_test).astype(float)))
)
test_lgbm = (
    test.select(['store_nbr', 'date', TARGET_COL]).rename({TARGET_COL: 'y_true'})
    .with_columns(pl.Series('pred_lgbm', model_lgbm.predict(X_test).astype(float)))
)
test_xgb.write_parquet(RESULTS / 'test_xgboost.parquet')
test_lgbm.write_parquet(RESULTS / 'test_lightgbm.parquet')
print('XGBoost & LightGBM ✓')

XGBoost Val           MAE=9.0  RMSE=14.7  MAPE=0.5%
LightGBM Val          MAE=9.7  RMSE=14.8  MAPE=0.6%
XGBoost & LightGBM ✓


# PatchTST & NHITS

In [5]:
min_required_len = LOOKBACK + HORIZON

# 2. Zu kurze Stores direkt in Polars herausfiltern
# Wir zählen die Einträge pro Store und behalten nur die, die lang genug sind
train_val_clean = (
    train_val
    .filter(pl.len().over("store_nbr") >= min_required_len)
)

# Optional: Kurzes Feedback im Terminal, falls Stores rausgeflogen sind
ursprung_stores = train_val["store_nbr"].n_unique()
bereinigte_stores = train_val_clean["store_nbr"].n_unique()
if ursprung_stores != bereinigte_stores:
    print(f"⚠️ {ursprung_stores - bereinigte_stores} Store(s) wurden entfernt, da sie weniger als {min_required_len} Datenpunkte hatten.")

# 3. Formate mit den bereinigten Daten erstellen
train_val_nf = to_nf_format(train_val_clean)

⚠️ 1 Store(s) wurden entfernt, da sie weniger als 412 Datenpunkte hatten.


In [6]:


static_df = (
    train_val_clean.select(['store_nbr'] + STAT_EXOG)
                   .unique(subset=['store_nbr']).sort('store_nbr')
                   .with_columns(pl.col('store_nbr').cast(pl.Utf8).alias('unique_id'))
                   .select(['unique_id'] + STAT_EXOG)
                   .with_columns([pl.col(c).cast(pl.Float32) for c in STAT_EXOG])
                   .to_pandas()
)


# PatchTST Modellierung
print('PatchTST...')
nf_pt = NeuralForecast(models=[PatchTST(
    h=HORIZON, input_size=LOOKBACK, patch_len=16, stride=8,
    encoder_layers=2, n_heads=8, hidden_size=64, linear_hidden_size=128,
    dropout=0.2, fc_dropout=0.2, scaler_type='standard',
    max_steps=200, batch_size=64, learning_rate=1e-4,
    early_stop_patience_steps=10, val_check_steps=25, random_seed=42,
)], freq='D')

# Da train_val_nf nun sauber ist, läuft fit() jetzt ohne ValueError durch
nf_pt.fit(df=train_val_nf[['unique_id', 'ds', 'y']], val_size=HORIZON)

test_patchtst = nf_to_polars(nf_pt.predict(), 'PatchTST', 'pred_patchtst', test)
test_patchtst.write_parquet(RESULTS / 'test_patchtst.parquet')
print('PatchTST ✓')

Seed set to 42


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.

  | Name         | Type              | Params | Mode  | FLOPs
-------------------------------------------------------------------
0 | loss         | MAE               | 0      | train | 0    
1 | padder_train | ConstantPad1d     | 0      | train | 0    
2 | scaler       | TemporalNorm      | 0      | train | 0    
3 | model        | PatchTST_backbone | 275 K  | train | 0    
-------------------------------------------------------------------
275 K     Trainable params
2         Non-trainable params
275 K     Total params
1.100     Total estimated model params size (MB)
65        Modules in train mode
0         Modules in eval mode
0         Total Flops


PatchTST...


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=200` reached.
Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

PatchTST ✓


In [8]:
# NHITS
print('NHITS...')
required_cols_nhits = ['unique_id', 'ds', 'y'] + HIST_EXOG
nf_nh = NeuralForecast(models=[NHITS(
    h=HORIZON, input_size=LOOKBACK,
    stat_exog_list=STAT_EXOG, hist_exog_list=HIST_EXOG,
    scaler_type='standard', max_steps=200, batch_size=128,
    learning_rate=1e-3, early_stop_patience_steps=5,
    val_check_steps=10, random_seed=42,
)], freq='D')
nf_nh.fit(df=train_val_nf[required_cols_nhits], static_df=static_df, val_size=len(test['date'].unique()))
test_nhits = nf_to_polars(nf_nh.predict(), 'NHITS', 'pred_nhits', test)
test_nhits.write_parquet(RESULTS / 'test_nhits.parquet')
print('NHITS ✓')

Seed set to 42
c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\neuralforecast\tsdataset.py:129: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  x = torch.from_numpy(x)
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.

  | Name         | Type          | Params | Mode  | FLOPs
-----------------------------

NHITS...


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
c:\Users\maxkr\AppData\Local\Programs\Python\Python312\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

NHITS ✓
